**Notebook 05a: Extract SGLT2 prescribing from the EPD files**

The 18 monthly EPD files are too large to analyse directly (about 7 GB each). This notebook reads each file in chunks and keeps only the rows needed: SGLT2 inhibitor items prescribed by the 465 practices in `practice_master_lookup.csv`. The result is a small file used by notebook 05.

Input: the 18 files in `EPD_Data` and `practice_master_lookup.csv` (from notebook 03).

Output: `epd_sglt2_extract.csv`. Each run replaces the old version.

This notebook is slow. Expect 30 to 90 minutes in total. Run it once, and keep the saved outputs so the checks can be reviewed.

**Step 1: Load libraries and settings**

Practices are selected by the master practice list. If a file has an ICB code column, rows from the three ICBs are also kept, so practices missing from the 2024/25 list can be spotted later. SGLT2 inhibitors are picked by chemical name, because every one of them ends in "gliflozin" (dapagliflozin, empagliflozin, canagliflozin, ertugliflozin and their combinations).

In [1]:
import os
import time
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

master = pd.read_csv('practice_master_lookup.csv')
MASTER_CODES = set(master['PRACTICE_CODE'])
ICB_CODES = {'QHL', 'QUA', 'QWU'}

epd_files = sorted(f for f in os.listdir('EPD_Data') if f.lower().endswith('.csv'))
print('Practices in master list:', len(MASTER_CODES))
print('EPD files found:', len(epd_files))
for i, f in enumerate(epd_files, 1):
    print(i, f)

Practices in master list: 465
EPD files found: 18
1 epd_snomed_202501.csv
2 epd_snomed_202502.csv
3 epd_snomed_202503.csv
4 epd_snomed_202504.csv
5 epd_snomed_202505.csv
6 epd_snomed_202506.csv
7 epd_snomed_202507.csv
8 epd_snomed_202508.csv
9 epd_snomed_202509.csv
10 epd_snomed_202510.csv
11 epd_snomed_202511.csv
12 epd_snomed_202512.csv
13 epd_snomed_202601.csv
14 epd_snomed_202602.csv
15 epd_snomed_202603.csv
16 epd_snomed_202604.csv
17 epd_snomed_202605.csv
18 epd_snomed_202606.csv


**Step 2: Check the column names in every file**

The first two files (Jan and Feb 2025) use different column names from the rest. The table shows which of the columns we need appear in each file. Columns are listed down the side and the files across the top, numbered as in Step 1. `Y` means present and `-` means absent. `TOTAL_QUANTITY` and `ICB_CODE` are the two that matter most.

In [2]:
KEY_COLS = ['YEAR_MONTH', 'PRACTICE_CODE', 'PRACTICE_NAME', 'ICB_CODE',
            'BNF_CHEMICAL_SUBSTANCE_CODE', 'BNF_CHEMICAL_SUBSTANCE',
            'CHEMICAL_SUBSTANCE_BNF_DESCR', 'BNF_PRESENTATION_CODE', 'BNF_CODE',
            'BNF_PRESENTATION_NAME', 'BNF_DESCRIPTION',
            'ITEMS', 'TOTAL_QUANTITY', 'NIC', 'ACTUAL_COST']

header_check = {}
for i, f in enumerate(epd_files, 1):
    cols = pd.read_csv(os.path.join('EPD_Data', f), nrows=0).columns.tolist()
    header_check[i] = ['Y' if c in cols else '-' for c in KEY_COLS]

print(pd.DataFrame(header_check, index=KEY_COLS).to_string())

                             1  2  3  4  5  6  7  8  9  10 11 12 13 14 15 16 17 18
YEAR_MONTH                    Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y
PRACTICE_CODE                 Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y
PRACTICE_NAME                 Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y
ICB_CODE                      Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y
BNF_CHEMICAL_SUBSTANCE_CODE   -  -  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y
BNF_CHEMICAL_SUBSTANCE        Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y
CHEMICAL_SUBSTANCE_BNF_DESCR  Y  Y  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -
BNF_PRESENTATION_CODE         -  -  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y
BNF_CODE                      Y  Y  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -
BNF_PRESENTATION_NAME         -  -  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y  Y
BNF_DESCRIPTION               Y  Y  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -
ITEM

**Step 3: Define the extraction**

For each file the code works out which format it is, renames the columns to one standard set, reads the file in chunks of 500,000 rows and keeps the SGLT2 rows from the chosen practices. If `TOTAL_QUANTITY` or `ICB_CODE` is missing from a file, the column is filled with blanks, so the check in Step 2 shows what is missing.

In [3]:
def column_map(cols):
    """Map the column names in a file to one standard set of names."""
    m = {'YEAR_MONTH': 'YEAR_MONTH', 'PRACTICE_CODE': 'PRACTICE_CODE',
         'PRACTICE_NAME': 'PRACTICE_NAME', 'ITEMS': 'ITEMS',
         'NIC': 'NIC', 'ACTUAL_COST': 'ACTUAL_COST'}
    if 'BNF_CHEMICAL_SUBSTANCE_CODE' in cols:        # newer format (Mar 2025 onwards)
        m.update({'BNF_CHEMICAL_SUBSTANCE_CODE': 'BNF_CHEMICAL_SUBSTANCE_CODE',
                  'BNF_CHEMICAL_SUBSTANCE': 'BNF_CHEMICAL_SUBSTANCE',
                  'BNF_PRESENTATION_CODE': 'BNF_PRESENTATION_CODE',
                  'BNF_PRESENTATION_NAME': 'BNF_PRESENTATION_NAME'})
    else:                                             # older format (Jan and Feb 2025)
        m.update({'BNF_CHEMICAL_SUBSTANCE': 'BNF_CHEMICAL_SUBSTANCE_CODE',
                  'CHEMICAL_SUBSTANCE_BNF_DESCR': 'BNF_CHEMICAL_SUBSTANCE',
                  'BNF_CODE': 'BNF_PRESENTATION_CODE',
                  'BNF_DESCRIPTION': 'BNF_PRESENTATION_NAME'})
    for optional in ['ICB_CODE', 'TOTAL_QUANTITY']:
        if optional in cols:
            m[optional] = optional
    return m


FINAL_COLS = ['YEAR_MONTH', 'PRACTICE_CODE', 'PRACTICE_NAME', 'ICB_CODE',
              'BNF_CHEMICAL_SUBSTANCE_CODE', 'BNF_CHEMICAL_SUBSTANCE',
              'BNF_PRESENTATION_CODE', 'BNF_PRESENTATION_NAME',
              'ITEMS', 'TOTAL_QUANTITY', 'NIC', 'ACTUAL_COST']


def extract_file(path):
    cols = pd.read_csv(path, nrows=0).columns.tolist()
    m = column_map(cols)
    missing = [c for c in m if c not in cols]
    if missing:
        raise ValueError(f'{os.path.basename(path)} is missing columns: {missing}')

    dtypes = {c: str for c in ['PRACTICE_CODE', 'ICB_CODE'] if c in cols}
    parts = []
    for chunk in pd.read_csv(path, usecols=list(m.keys()), dtype=dtypes, chunksize=500000):
        chunk = chunk.rename(columns=m)
        keep = chunk['PRACTICE_CODE'].isin(MASTER_CODES)
        if 'ICB_CODE' in chunk.columns:
            keep = keep | chunk['ICB_CODE'].isin(ICB_CODES)
        sub = chunk[keep]
        sub = sub[sub['BNF_CHEMICAL_SUBSTANCE'].str.contains('gliflozin', case=False, na=False)]
        if len(sub):
            parts.append(sub)

    out = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=list(m.values()))
    for c in FINAL_COLS:
        if c not in out.columns:
            out[c] = pd.NA
    out = out[FINAL_COLS].copy()
    out['SOURCE_FILE'] = os.path.basename(path)
    return out

print('Extraction function defined')

Extraction function defined


**Step 4: Run the extraction on all files**

This is the slow step. It prints one line per file with the number of rows kept and the time taken. If it stops partway, the error message names the file.

In [4]:
all_parts = []
start = time.time()
for i, f in enumerate(epd_files, 1):
    t = time.time()
    part = extract_file(os.path.join('EPD_Data', f))
    all_parts.append(part)
    print(f'{i}/{len(epd_files)}  {f}: {len(part)} rows kept, {time.time() - t:.0f} seconds')

epd = pd.concat(all_parts, ignore_index=True)
print(f'\nTotal rows: {len(epd)} | total time: {(time.time() - start) / 60:.1f} minutes')

1/18  epd_snomed_202501.csv: 7067 rows kept, 46 seconds
2/18  epd_snomed_202502.csv: 6878 rows kept, 45 seconds
3/18  epd_snomed_202503.csv: 6969 rows kept, 49 seconds
4/18  epd_snomed_202504.csv: 7031 rows kept, 49 seconds
5/18  epd_snomed_202505.csv: 7117 rows kept, 50 seconds
6/18  epd_snomed_202506.csv: 7024 rows kept, 51 seconds
7/18  epd_snomed_202507.csv: 7250 rows kept, 47 seconds
8/18  epd_snomed_202508.csv: 7007 rows kept, 45 seconds
9/18  epd_snomed_202509.csv: 7199 rows kept, 46 seconds
10/18  epd_snomed_202510.csv: 7312 rows kept, 46 seconds
11/18  epd_snomed_202511.csv: 7071 rows kept, 45 seconds
12/18  epd_snomed_202512.csv: 7393 rows kept, 46 seconds
13/18  epd_snomed_202601.csv: 7349 rows kept, 45 seconds
14/18  epd_snomed_202602.csv: 7123 rows kept, 43 seconds
15/18  epd_snomed_202603.csv: 7223 rows kept, 45 seconds
16/18  epd_snomed_202604.csv: 7219 rows kept, 45 seconds
17/18  epd_snomed_202605.csv: 7141 rows kept, 44 seconds
18/18  epd_snomed_202606.csv: 6894 rows 

**Step 5: Create a date column and flag the practices**

`YEAR_MONTH` appears in different styles across the files, for example 202501 or 2025-01. This step turns it into a proper date. The `in_master` flag shows whether the practice is in the 465-practice list.

In [5]:
def to_date(x):
    s = str(x)
    if s.isdigit():
        return pd.to_datetime(s[:4] + '-' + s[4:6] + '-01')
    return pd.to_datetime(s[:7] + '-01')

epd['date'] = epd['YEAR_MONTH'].apply(to_date)
epd['in_master'] = epd['PRACTICE_CODE'].isin(MASTER_CODES)

print('Date range:', epd['date'].min().date(), 'to', epd['date'].max().date())
print('Rows from practices in the master list:', epd['in_master'].sum())
print('Rows from other practices in the three ICBs:', (~epd['in_master']).sum())

Date range: 2025-01-01 to 2026-06-01
Rows from practices in the master list: 127368
Rows from other practices in the three ICBs: 899


**Step 6: Check the extract**

These checks cover months, chemicals, dapagliflozin presentations and missing values. They are the main things to review before notebook 05. The presentation table matters most, because it shows how branded Forxiga and generic dapagliflozin appear in the data.

In [6]:
print('Rows, practices and items by month:')
by_month = epd.groupby('date').agg(rows=('ITEMS', 'size'),
                                   practices=('PRACTICE_CODE', 'nunique'),
                                   items=('ITEMS', 'sum'))
print(by_month.to_string())

print('\nItems by chemical:')
print(epd.groupby('BNF_CHEMICAL_SUBSTANCE')['ITEMS'].sum()
      .sort_values(ascending=False).to_string())

print('\nDapagliflozin presentations (all months):')
dapa = epd[epd['BNF_CHEMICAL_SUBSTANCE'].str.contains('dapagliflozin', case=False, na=False)]
print(dapa.groupby('BNF_PRESENTATION_NAME')[['ITEMS', 'TOTAL_QUANTITY']].sum()
      .sort_values('ITEMS', ascending=False).head(25).to_string())

print('\nMissing values:')
print(epd[FINAL_COLS].isnull().sum().to_string())

Rows, practices and items by month:
            rows  practices   items
date                               
2025-01-01  7067        490   81677
2025-02-01  6878        490   73594
2025-03-01  6969        494   78350
2025-04-01  7031        491   81936
2025-05-01  7117        494   85033
2025-06-01  7024        494   82492
2025-07-01  7250        485   91512
2025-08-01  7007        494   84775
2025-09-01  7199        491   91496
2025-10-01  7312        491   94116
2025-11-01  7071        490   86437
2025-12-01  7393        485   97978
2026-01-01  7349        496   93485
2026-02-01  7123        495   87539
2026-03-01  7223        493   96984
2026-04-01  7219        492   95901
2026-05-01  7141        494   95222
2026-06-01  6894        490  101965

Items by chemical:
BNF_CHEMICAL_SUBSTANCE
Dapagliflozin                1006293
Empagliflozin                 477237
Canagliflozin                  66502
Empagliflozin/metformin        29001
Dapagliflozin/metformin        18328
Canagliflozin/me

**Step 7: Save the extract**

The file is small enough to share, and notebook 05 reads it. The raw EPD files are not needed again.

In [7]:
epd.to_csv('epd_sglt2_extract.csv', index=False)

size_mb = os.path.getsize('epd_sglt2_extract.csv') / 1e6
print('Saved epd_sglt2_extract.csv:', epd.shape, f'| {size_mb:.1f} MB')

Saved epd_sglt2_extract.csv: (128267, 15) | 22.5 MB


**Acknowledgement of AI assistance**

This notebook was written by Mifta Mahmood with assistance from Claude (Anthropic), a generative AI assistant, which helped to write and debug the code. All code was run, checked and verified by the author, and the outputs shown are from the author's own runs. The final decisions on data, methods and interpretation are the author's own.